# Stream C - Evidence Builder (v3, decircularized)

Builds an offline, reproducible evidence table from independent sources.

**Key changes vs. previous version:**
1. **Caption is NEVER included in `evidence_text`** (removes circularity).
2. VERITE rows now use `VERITE_articles.csv` body text as evidence.
3. NewsCLIPpings rows use article title + topic + source + timestamp as evidence (no caption).
4. MMFakeBench rows use text_source + image_source + fake_cls metadata as evidence.

This evidence is then independent of the caption, so cosine/NLI similarity becomes a meaningful signal.


In [1]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES, RAW

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [2]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
BATCH_SIZE = 256


Torch: 2.6.0+cu124
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Ti
VRAM GB: 8.0


In [3]:
# === Load manifest ===
EVIDENCE_DIR = STREAM_C_CACHE / "evidence"
EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ALL_MANIFEST, dtype={
    "pair_id":"string","dataset":"string","split":"string","caption":"string",
    "title":"string","topic":"string","source":"string","timestamp":"string",
    "raw_image_path":"string","text_source":"string","image_source":"string","fake_cls":"string"
}, low_memory=False)
print("Rows:", len(df))
display(df.groupby(["dataset", "split"]).size().reset_index(name="rows"))


Rows: 97361


,dataset,split,rows
0,mmfakebench,test,10000
1,mmfakebench,val,1000
2,newsclippings,test,7264
3,newsclippings,train,71072
4,newsclippings,val,7024
5,verite,test,1001


In [4]:
# === Load VERITE_articles for independent evidence ===
# VERITE_articles.csv contains the actual article body associated with each VERITE row.
# This is INDEPENDENT of the caption and is the proper evidence source for VERITE.
VERITE_ARTICLES_PATH = RAW / "verite" / "image-text-verification" / "VERITE" / "VERITE_articles.csv"
verite_articles_lookup = {}
if VERITE_ARTICLES_PATH.exists():
    va = pd.read_csv(VERITE_ARTICLES_PATH, low_memory=False)
    print("VERITE_articles columns:", list(va.columns))
    print("VERITE_articles rows:", len(va))
    # Build a robust lookup: try common id columns. We try multiple keys because the
    # VERITE CSV schema can vary slightly across releases.
    candidate_keys = ["id", "Id", "image_id", "article_id", "index", "Unnamed: 0"]
    key_col = None
    for c in candidate_keys:
        if c in va.columns:
            key_col = c
            break
    # Best body column: prefer 'article' / 'body' / 'text' / 'content'
    body_candidates = ["article", "body", "text", "content", "article_body", "full_text", "query", "true_caption"]
    body_col = None
    for c in body_candidates:
        if c in va.columns:
            body_col = c
            break
    print(f"Using key_col={key_col}, body_col={body_col}")
    if key_col and body_col:
        for _, row in va.iterrows():
            k = str(row[key_col]).strip()
            v = str(row[body_col]) if pd.notna(row[body_col]) else ""
            if k and v:
                verite_articles_lookup[k] = v
        print(f"Built VERITE article lookup: {len(verite_articles_lookup)} entries")
    else:
        print("WARNING: could not identify key/body columns in VERITE_articles.csv")
        print("Available columns for manual inspection:", list(va.columns))
else:
    print(f"WARNING: {VERITE_ARTICLES_PATH} not found. VERITE rows will fall back to metadata-only evidence.")

print()
print("Sample VERITE article (if any):")
if verite_articles_lookup:
    sk = next(iter(verite_articles_lookup))
    sv = verite_articles_lookup[sk]
    print(f"  key={sk!r}")
    print(f"  body[:200]={sv[:200]!r}")


VERITE_articles columns: ['Unnamed: 0', 'id', 'true_url', 'false_caption', 'true_caption', 'false_url', 'query', 'snopes_url']
VERITE_articles rows: 338
Using key_col=id, body_col=query
Built VERITE article lookup: 325 entries

Sample VERITE article (if any):
  key='0'
  body[:200]='shenzhen police arresting during covid'


In [5]:
# === Build evidence_text WITHOUT the caption ===
# Critical rule: evidence_text must be independent of the caption.
# Including the caption would make stance similarity trivially high for all rows
# regardless of whether the image-text pair is genuine or out-of-context.

def clean_text(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x)).strip()

def truncate(s, max_chars=2000):
    # NLI/SBERT models tokenize at ~512 tokens; 2000 chars is a safe cap.
    if not s:
        return ""
    return s[:max_chars]

def lookup_verite_article(row):
    # Try to match VERITE row to its article body using several possible keys.
    # We try original_id, source_text_id, source_image_id, raw_image_path stem, pair_id.
    for k_field in ("original_id", "source_text_id", "source_image_id", "pair_id"):
        v = clean_text(row.get(k_field, ""))
        if v and v in verite_articles_lookup:
            return verite_articles_lookup[v]
    # Fallback: try basename of raw_image_path without extension
    rip = clean_text(row.get("raw_image_path", ""))
    if rip:
        stem = Path(rip).stem
        if stem in verite_articles_lookup:
            return verite_articles_lookup[stem]
    return ""

def build_evidence_text(row):
    """Build evidence text that is INDEPENDENT of the caption.

    Per dataset:
    - VERITE: prefer VERITE_articles body text. If unavailable, use raw_image_path + dataset record.
    - NewsCLIPpings: use title + topic + source + timestamp.
    - MMFakeBench: use text_source + image_source + fake_cls + raw_image_path.
    """
    parts = []
    dataset = clean_text(row.get("dataset", ""))

    if dataset == "verite":
        article = lookup_verite_article(row)
        if article:
            parts.append(f"Article body: {truncate(article)}")
        # Add raw image reference as a weak but independent signal
        rip = clean_text(row.get("raw_image_path", ""))
        if rip:
            parts.append(f"Raw image reference: {rip}")

    elif dataset == "newsclippings":
        title = clean_text(row.get("title", ""))
        topic = clean_text(row.get("topic", ""))
        source = clean_text(row.get("source", ""))
        timestamp = clean_text(row.get("timestamp", ""))
        if title:
            parts.append(f"Article title: {title}")
        if topic or source or timestamp:
            parts.append(f"News metadata: topic={topic}; source={source}; timestamp={timestamp}")
        rip = clean_text(row.get("raw_image_path", ""))
        if rip:
            parts.append(f"Raw image reference: {rip}")

    elif dataset == "mmfakebench":
        text_source = clean_text(row.get("text_source", ""))
        image_source = clean_text(row.get("image_source", ""))
        fake_cls = clean_text(row.get("fake_cls", ""))
        if text_source or image_source or fake_cls:
            parts.append(f"MMFakeBench metadata: text_source={text_source}; image_source={image_source}; fake_cls={fake_cls}")
        rip = clean_text(row.get("raw_image_path", ""))
        if rip:
            parts.append(f"Raw image reference: {rip}")

    # Always include dataset record as the lightest possible fallback
    parts.append(f"Dataset record: dataset={dataset}; split={clean_text(row.get('split', ''))}")
    return " | ".join(parts)

evidence = df[["pair_id", "dataset", "split", "label", "caption"]].copy()
evidence["evidence_text"] = df.apply(build_evidence_text, axis=1)
evidence["evidence_len"] = evidence["evidence_text"].str.len()
evidence["caption_len"] = evidence["caption"].fillna("").astype(str).str.len()

# Sanity check: confirm no caption appears verbatim inside evidence_text
# Sanity check: confirm no caption appears as a LABELED field in evidence_text.
# Substring overlap with article title is fine (titles legitimately contain captions);
# what we're guarding against is the old bug where "Caption: {caption}" was injected.
def _has_labeled_caption_leak(row):
    ev = (row["evidence_text"] or "")
    return "Caption:" in ev  # the old bug's signature

leak_count = int(evidence.apply(_has_labeled_caption_leak, axis=1).sum())
print(f"Labeled-caption leak count (should be 0): {leak_count}")
assert leak_count == 0, "Circularity detected: 'Caption:' field is being injected into evidence_text"

# Informational: how many rows have caption as a substring of evidence (incidental, OK)
def _caption_substring_overlap(row):
    cap = (row["caption"] or "").strip()
    ev = (row["evidence_text"] or "")
    if not cap or len(cap) < 20:
        return False
    return cap[:50] in ev
incidental = int(evidence.apply(_caption_substring_overlap, axis=1).sum())
print(f"Incidental caption/title overlap (informational, OK): {incidental}")

evidence.to_csv(EVIDENCE_DIR / "raw_evidence_all.csv", index=False)
pickle.dump(evidence, open(EVIDENCE_DIR / "raw_evidence_all.pkl", "wb"), protocol=4)
for (dataset, split), sub in evidence.groupby(["dataset", "split"]):
    sub.to_csv(EVIDENCE_DIR / f"raw_evidence_{dataset}_{split}.csv", index=False)
display(evidence.head())


Labeled-caption leak count (should be 0): 0
Incidental caption/title overlap (informational, OK): 25


,pair_id,dataset,split,label,caption,evidence_text,evidence_len,caption_len
0,nc_620701_0,newsclippings,train,0,Saudi troops cheer as they ride at the back of...,"Article title: Inside Mazrak refugee camp, Yem...",250,125
1,nc_620701_1,newsclippings,train,1,Israeli soldiers ride on a tank to a position ...,Article title: Photo highlights of the day | N...,243,73
2,nc_1569820_0,newsclippings,train,0,Tiffany is north east of Aberdeen,Article title: Improvement call for CNR North ...,280,33
3,nc_1569820_1,newsclippings,train,1,Some ferry services were affected between Holy...,Article title: Warning of 'severe weather' to ...,275,71
4,nc_1490080_0,newsclippings,train,0,Rock band Paramore is another group which sell...,Article title: Whatever floats your boat - the...,282,55


In [6]:
# === Summary ===
summary = evidence.groupby(["dataset", "split"]).agg(
    rows=("pair_id", "count"),
    mean_evidence_len=("evidence_len", "mean"),
    mean_caption_len=("caption_len", "mean")
).reset_index()
summary.to_csv(TABLES / "stream_c_evidence_summary.csv", index=False)
display(summary)

# Per-dataset coverage of high-quality evidence (i.e., article body or article title present)
def _is_high_quality(s):
    s = s or ""
    return ("Article body:" in s) or ("Article title:" in s)

evidence["high_quality"] = evidence["evidence_text"].fillna("").astype(str).apply(_is_high_quality)
hq = evidence.groupby(["dataset", "split"])["high_quality"].mean().reset_index()
hq["high_quality_pct"] = (hq["high_quality"] * 100).round(2)
display(hq[["dataset", "split", "high_quality_pct"]])

print("Evidence builder v3 complete.")


,dataset,split,rows,mean_evidence_len,mean_caption_len
0,mmfakebench,test,10000,229.172100,79.623900
1,mmfakebench,val,1000,223.354000,79.030000
2,newsclippings,test,7264,280.159967,97.778634
3,newsclippings,train,71072,281.738758,98.137748
4,newsclippings,val,7024,279.451737,97.401054
5,verite,test,1001,99.516484,125.358641


,dataset,split,high_quality_pct
0,mmfakebench,test,0.00
1,mmfakebench,val,0.00
2,newsclippings,test,99.99
3,newsclippings,train,99.97
4,newsclippings,val,99.99
5,verite,test,32.47


Evidence builder v3 complete.
